# Quantum Time Series Analysis: Reports
Compatible with Qiskit 1.2.4+

### Author
- Jacob L. Cybulski, Enquanted

### Date
- Feb 2025: Started

### Aims
> *This script aims to compare results obtained from running different quantum time series models.*

In [1]:
import sys
sys.path.append('.')
sys.path

['/home/jacob/miniconda3/envs/qiskit-gpu/lib/python311.zip',
 '/home/jacob/miniconda3/envs/qiskit-gpu/lib/python3.11',
 '/home/jacob/miniconda3/envs/qiskit-gpu/lib/python3.11/lib-dynload',
 '',
 '/home/jacob/miniconda3/envs/qiskit-gpu/lib/python3.11/site-packages',
 '.']

In [2]:
import os
import numpy as np
import pylab
import time
import copy
import pandas as pd
from tqdm.notebook import tqdm

from IPython.display import clear_output

from utils.Window import *
from utils.Charts import *
from utils.Files import *

import matplotlib.pyplot as plt
from matplotlib import set_loglevel
set_loglevel("error")
%matplotlib inline

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
### Listing control
debug = True
seed = 2022

### Software version
MAJOR = 9
MINOR = 12

### Constants
LOG_NAME = 'log_4_iter'
DATA_NAME = '2_sins'
DATA_PATH = f'{LOG_NAME}/data'
TRAIN_PATH = f'{LOG_NAME}/training'
ANALYSIS_PATH = f'{LOG_NAME}/analysis'
FIGURES_PATH = f'{LOG_NAME}/figures'
REPORTS_PATH = f'{LOG_NAME}/reports'

### Show constants
(LOG_NAME, DATA_NAME,DATA_PATH, TRAIN_PATH, ANALYSIS_PATH, FIGURES_PATH, REPORTS_PATH)

('log_4_iter',
 '2_sins',
 'log_4_iter/data',
 'log_4_iter/training',
 'log_4_iter/analysis',
 'log_4_iter/figures',
 'log_4_iter/reports')

## Utilities

In [4]:
def simple_report(df):

    # Extract the relevant info
    df.columns = df.columns.str.replace('_', '')
    df = df.replace({'_': '-'}, regex=True)
    report_data = df['Data'][0]
    report_model = df['Model'][0]
    report_epochs = df['Epochs'][0]
    report_label = f'tab:{report_data}-{report_model}'
    df = df.drop(['Data', 'Model', 'TrMedMAE', 'TsMedMAE'], axis=1)
    df = df[['Qubits', 'Params', 
             'TrMedR2', 'TrMedMSE', 'TsMedR2', 'TsMedMSE', 'Specs']]
    df.columns = ['Qubits', 'Params', 'TR2', 'TMSE', 'VR2', 'VMSE', 'Specs']
    vspace = '\\vspace{2mm}'

    report = df.style.hide().format(precision=4).to_latex(position_float='centering', 
        label=report_label, hrules=True, 
        column_format='rr@{\hskip 8pt}cc@{\hskip 8pt}cc@{\hskip 8pt}l',
        caption=f'Median model performance ({report_data}-{report_model}, ep={report_epochs}{vspace})')
    return report

In [5]:
def flex_report(df, position='t', caption=None, label=None, tab='    ', 
                precision=4, cformat=None, calias=None, hlines=[], add_epoch=False):
    
    # Extract the relevant info
    df.columns = df.columns.str.replace('_', '')
    df = df.replace({'_': '-'}, regex=True)
    mtype = df['Type'][0]
    model = df['Model'][0]
    epochs = df['Epochs'][0]
    df = df.drop(['TrMedMAE', 'TrMedMSE', 'TsMedMAE', 'TsMedMSE'], axis=1)
    df = df[['Type', 'Model', 'Qubits', 'Params', 'TrMedR2', 'TsMedR2', 'Specs']]
    cols = len(df.columns)
    rows = len(df)

    # Calculate defaults
    if cformat is None:
        cformat = "@{\\extracolsep{4pt}}rrrr@{\\hskip 4pt}r@{\\hskip 4pt}r@{\\hskip 4pt}l"

    if caption is None:
        caption = f'Compararison of model performance'

    if label is None:
        label = f'tab:models-comparison'

    if calias is None:
        calias = ['Model', 'Type', 'Qubits', 'Params', 'R2', 'R2', 'Specs']

    # Generate report
    rp = ''

    rp += "\\begin{table}"+f"[{position}]\n"
    rp += "\\vspace{-5mm}\n"
    rp += "\\begin{center}\n"
    rp += "\\caption{"+f"{caption}"+"}\n"
    rp += "\\label{"+f"{label}"+"}\n"
    rp += "\\scriptsize\n"
    rp += "\\vspace{2mm}"
    rp += "\\begin{tabular}{ "+f"{cformat}"+" }\n"
    rp +=    f"{tab}"+"\\hline\n"
    rp +=    f"{tab}"+"& & & & {Training} & {Testing} & \\\\\n"
    #rp +=    f"{tab}"+"\\cline{3-4}\\cline{5-6}\n"
    rp +=    f"{tab}"+" & ".join(calias)+'\\\\\n'
    rp +=    f"{tab}"+"\\hline\\hline\n"

    for row in range(rows):
        rp += f"{tab}"
        rp += f"{df['Model'][row]}"+" & "
        rp += f"{df['Type'][row]}"+" & "
        rp += f"{df['Qubits'][row]}"+" & "
        rp += f"{df['Params'][row]}"+" & "
        rp += f"{df['TrMedR2'][row]: 0.{precision}f}"+" & "
        rp += f"{df['TsMedR2'][row]: 0.{precision}f}"+" & "
        rp += "\\text{"+f"{df['Specs'][row]}"+"}"
        rp += '\\\\\n'
        if row in hlines:
            rp += f"{tab}"+"\\hdashline[2pt/1pt]\n"

    rp +=    f"{tab}"+"\\hline\n"
    rp += "\\end{tabular}\n"
    rp += "\\end{center}\n"
    rp += "\\vspace{-2mm}\n"
    rp += "\\end{table}\n\n"

    return rp

## Identify all required reports

In [6]:
### Define the required report names (as per log)
report_fname_list = ['2_sins_serial', '2_sins_parallel', '2_sins_xparallel',
                     '2_sins_sw_xqnn_ng', '2_sins_sw_ovload', '2_sins_sw_cnn']
report_selected_lines = [[3, 4], [1, 5], [1], [8, 7], [5], [0]]
report_mid_lines = [4, 7]

## Produce reports for the selected models

In [7]:
### Generate reports for the selected models
dfs = []
for i in range(len(report_fname_list)):
    rf = report_fname_list[i]
    next_df = pd.read_csv(f'{REPORTS_PATH}/{rf}.tsv', delimiter='\t')
    dfs.append(next_df.iloc[report_selected_lines[i]])
df = pd.concat(dfs)
df = df.reset_index(drop=True)
df.loc[df['Data']=='2_sins', 'Data'] = 'pqft'
df.loc[df['Data']=='2_sins_sw', 'Data'] = 'forecast'
df = df.rename(columns={'Data': 'Type'})
df

,Type,Model,Specs,Qubits,Params,Epochs,Tr_Med_R2,Tr_Med_MSE,Tr_Med_MAE,Ts_Med_R2,Ts_Med_MSE,Ts_Med_MAE
0,pqft,serial,q1 l21,1,66,60,0.996592,1.633400e-04,0.009511,0.919247,2.516340e-03,0.033663
1,pqft,serial,q1 l27,1,84,60,0.997700,1.102500e-04,0.008621,0.910652,2.784140e-03,0.037652
2,pqft,parallel,q5 bl3 al3,5,120,60,0.845981,7.382680e-03,0.075095,0.695708,9.481980e-03,0.090330
3,pqft,parallel,q5 bl1 al3,5,90,60,0.814541,8.889690e-03,0.078616,0.713720,8.920740e-03,0.079516
4,pqft,xparallel,q3 bl7 al1,3,81,60,0.998035,9.417000e-05,0.007695,0.982269,5.525000e-04,0.016264
5,forecast,xqnn,q7 in5 fm1 anz4,7,105,60,0.985771,5.604000e-04,0.019080,0.847805,5.322660e-03,0.060938
6,forecast,xqnn,q7 in5 fm1 anz3,7,84,60,0.960313,1.563050e-03,0.031978,0.879622,4.209920e-03,0.048069
7,forecast,ovload,q4 xl3 il2,4,167,60,0.888959,4.373250e-03,0.056435,0.873451,4.425750e-03,0.054519
8,forecast,cnn,hl150 100 050,0,20800,60,1.000000,4.351835e-10,0.000017,1.000000,5.132879e-10,0.000019


In [8]:
report = flex_report(df, position='ht', hlines=report_mid_lines)
print(report)

\begin{table}[ht]
\vspace{-5mm}
\begin{center}
\caption{Compararison of model performance}
\label{tab:models-comparison}
\scriptsize
\vspace{2mm}\begin{tabular}{ @{\extracolsep{4pt}}rrrr@{\hskip 4pt}r@{\hskip 4pt}r@{\hskip 4pt}l }
    \hline
    & & & & {Training} & {Testing} & \\
    Model & Type & Qubits & Params & R2 & R2 & Specs\\
    \hline\hline
    serial & pqft & 1 & 66 &  0.9966 &  0.9192 & \text{q1 l21}\\
    serial & pqft & 1 & 84 &  0.9977 &  0.9107 & \text{q1 l27}\\
    parallel & pqft & 5 & 120 &  0.8460 &  0.6957 & \text{q5 bl3 al3}\\
    parallel & pqft & 5 & 90 &  0.8145 &  0.7137 & \text{q5 bl1 al3}\\
    xparallel & pqft & 3 & 81 &  0.9980 &  0.9823 & \text{q3 bl7 al1}\\
    \hdashline[2pt/1pt]
    xqnn & forecast & 7 & 105 &  0.9858 &  0.8478 & \text{q7 in5 fm1 anz4}\\
    xqnn & forecast & 7 & 84 &  0.9603 &  0.8796 & \text{q7 in5 fm1 anz3}\\
    ovload & forecast & 4 & 167 &  0.8890 &  0.8735 & \text{q4 xl3 il2}\\
    \hdashline[2pt/1pt]
    cnn & forecast & 0 & 2

## System

In [9]:
import os
import sys
print(f"\nOperating environment:\n")
os.system('lsb_release -d -s')
os.system('python --version')
print(f'Conda {os.path.basename(sys.prefix)}\n')


Operating environment:

Ubuntu 22.04.5 LTS
Python 3.11.11
Conda qiskit-gpu



In [10]:
print(f"\nSignificant Python packages:\n")
os.system('pip list | grep -e qiskit');


Significant Python packages:

qiskit                        1.2.4
qiskit-aer-gpu                0.15.1
qiskit-algorithms             0.3.1
qiskit-ibm-runtime            0.32.0
qiskit-machine-learning       0.8.1
qiskit-optimization           0.6.1
qiskit-sphinx-theme           1.16.1
